# SAbDab2 VHH scaffold 筛选复盘

本 Notebook 读取冻结后的筛选结果，复算统计与一致性检查。它不把‘结构模板合格’解释成‘已经结合 GLP‑1’。

In [1]:
# 所有路径相对于数据库根目录；避免把个人机器的绝对路径写入结果表。
from pathlib import Path
import json, sqlite3
import pandas as pd

ROOT = Path.cwd()
summary = json.loads((ROOT/'registry/database_summary.json').read_text())
instances = pd.read_csv(ROOT/'registry/antibody_instances.tsv', sep='\t', low_memory=False)
candidates = pd.read_csv(ROOT/'registry/scaffold_candidates.tsv', sep='\t')
residues = pd.read_csv(ROOT/'registry/residue_map.tsv', sep='\t', low_memory=False)
selected = pd.read_csv(ROOT/'registry/selected_scaffolds.tsv', sep='\t')
validation = pd.read_csv(ROOT/'registry/boltzgen_export_validation.tsv', sep='\t')
funnel = pd.read_csv(ROOT/'qc/screening_funnel.tsv', sep='\t')
exclusions = pd.read_csv(ROOT/'qc/exclusion_log.tsv', sep='\t')
summary['counts']

{'raw_instances': 4508,
 'unique_pdb': 2391,
 'unique_sabdab_id': 1324,
 'archive_cif_files': 2391,
 'metadata_qualified_instances': 1227,
 'hard_qc_pass_instances': 703,
 'best_instance_per_sabdab_id': 333,
 'unique_exact_framework': 324,
 'framework_clusters': 245,
 'selected_primary': 10,
 'selected_reserve': 2,
 'boltzgen_check_total': 12,
 'boltzgen_check_pass': 12,
 'boltzgen_check_fail': 0,
 'boltzgen_check_pending': 0}

## 原始数据粒度与主键

In [2]:
# INSTANCE 应是一行一个 antibody instance；SABDAB_ID 允许重复，代表同一变量域的多次结构观测。
pd.DataFrame({
    'metric':['rows','unique INSTANCE','unique PDB','unique SABDAB_ID'],
    'value':[len(instances), instances.INSTANCE.nunique(), instances.PDB.nunique(), instances.SABDAB_ID.nunique()]
})

,metric,value
0,rows,4508
1,unique INSTANCE,4508
2,unique PDB,2391
3,unique SABDAB_ID,1324


In [3]:
# 主键和筛选漏斗必须满足确定性对账条件。
assert instances.INSTANCE.is_unique
assert candidates.candidate_id.is_unique
assert selected.candidate_id.is_unique
assert selected.selection_rank.is_unique
assert not residues.duplicated(['candidate_id','imgt_position']).any()
assert funnel.remaining_count.is_monotonic_decreasing
assert int(funnel.iloc[-1].remaining_count) == len(selected)
print('主键与漏斗对账：PASS')

主键与漏斗对账：PASS


## 筛选漏斗

In [4]:
funnel.style.format({'remaining_count':'{:,}'})

,stage_order,stage,remaining_count
0,1,SAbDab2 SD-H antibody instances,"4,508"
1,2,camelid-origin VHH scope,"3,574"
2,3,X-ray,"2,198"
3,4,resolution ≤2.5 Å,"1,242"
4,5,metadata-qualified for structure QC,"1,227"
5,6,hard structure QC pass,703
6,7,best instance per SAbDab ID,333
7,8,unique exact framework,324
8,9,framework cluster representatives,245
9,10,selected primary + reserve,12


## 首个排除原因

In [5]:
reason_counts = (exclusions.query("first_exclusion_reason != 'SELECTED'")
                 .first_exclusion_reason.value_counts().rename_axis('reason').reset_index(name='count'))
assert reason_counts['count'].sum() + (exclusions.first_exclusion_reason=='SELECTED').sum() == len(instances)
reason_counts

,reason,count
0,首轮只采用 X-ray 结构,1376
1,分辨率缺失或高于 2.5 Å,956
2,不在 camelid-origin VHH 主面板范围,934
3,结构/编号/二硫键硬 QC 未通过,524
4,同一 SAbDab ID 有质量更优实例,370
5,通过但未进入 12 个多样化代表面板,233
6,同一 framework 结构簇已有代表,79
7,Rfree 高于 0.30,13
8,完全相同 framework 已有质量更优代表,9
9,链共享/融合构建体不进入首轮,2


## 结构 QC 结果

In [6]:
# 这里只统计进入 metadata-qualified pool 的结构实例。
candidates.groupby('hard_status', dropna=False).size().rename('count').reset_index()

,hard_status,count
0,FAIL,524
1,PASS,703


In [7]:
# 展示最常见硬失败类别；完整逐条证据保存在 qc/qc_results.tsv。
failed = candidates.query("hard_status == 'FAIL'").copy()
(failed.hard_reasons.fillna('').str.split(' | ', regex=False).explode()
 .str.split(':').str[0].value_counts().head(15).rename_axis('rule').reset_index(name='count'))

,rule,count
0,unresolved_sequence_gap,193
1,canonical_disulfide_distance_invalid,181
2,extra_disulfide_involves_design_region,171
3,missing_imgt_anchors,36
4,n_terminal_framework_truncated,30
5,residue_mean_occupancy_below_0.5,22
6,nonstandard_variable_residue,18
7,framework_backbone_incomplete,16
8,peptide_bond_break,11
9,c_terminal_framework_truncated,11


## 最终 scaffold 面板

In [8]:
columns = ['selection_rank','role','candidate_id','pdb_code','sabdab_id','heavy_species',
           'resolution_a','variable_length_aa','cdr3_length_aa','quality_score',
           'framework_cluster_id','canonical_disulfide_rcsb_crosschecked','benchmark_7xl0',
           'boltzgen_check_status']
selected[columns].sort_values('selection_rank')

,selection_rank,role,candidate_id,pdb_code,sabdab_id,heavy_species,resolution_a,variable_length_aa,cdr3_length_aa,quality_score,framework_cluster_id,canonical_disulfide_rcsb_crosschecked,benchmark_7xl0,boltzgen_check_status
0,1,PRIMARY,pdb_00007xl0-A,7XL0,sabdab2_H03JML0000,lama glama,1.700,121,15,0.809943,FWC0109,True,True,PASS
1,2,PRIMARY,pdb_00006apo-A,6APO,sabdab2_H006BL0000,lama glama,1.168,116,11,0.917450,FWC0077,True,False,PASS
2,3,PRIMARY,pdb_00008v9x-A,8V9X,sabdab2_H04KPL0000,camelidae,1.550,113,7,0.836873,FWC0211,True,False,PASS
3,4,PRIMARY,pdb_00006xxo-A,6XXO,sabdab2_H02GEL0000,camelus dromedarius,1.500,124,17,0.861320,FWC0115,True,False,PASS
4,5,PRIMARY,pdb_00005l21-B,5L21,sabdab2_H01GHL0000,vicugna pacos,1.680,118,5,0.827825,FWC0060,True,False,PASS
5,6,PRIMARY,pdb_00008fq7-A,8FQ7,sabdab2_H05V6L0000,camelus dromedarius,1.400,118,11,0.849080,FWC0180,True,False,PASS
6,7,PRIMARY,pdb_00008e2n-B,8E2N,sabdab2_H0446L0000,vicugna pacos,1.100,127,17,0.874885,FWC0168,True,False,PASS
7,8,PRIMARY,pdb_00006xym-A,6XYM,sabdab2_H05T2L0000,lama glama,1.200,137,19,0.821417,FWC0117,True,False,PASS
8,9,PRIMARY,pdb_00008im0-B,8IM0,sabdab2_H04CVL0000,camelus bactrianus,1.310,121,15,0.881235,FWC0191,True,False,PASS
9,10,PRIMARY,pdb_00003tpk-A,3TPK,sabdab2_H05V3L0000,camelidae,1.300,120,14,0.879547,FWC0017,True,False,PASS


In [9]:
# 入选面板的结构包必须完整存在。
required_files = ['scaffold.cif','scaffold.yaml','residue_mapping.tsv','curation.json','qc.json']
missing = []
for package in selected.package_path:
    for name in required_files:
        if not (ROOT/package/name).is_file(): missing.append(f'{package}/{name}')
assert not missing, missing
actual_packages = {str(path.relative_to(ROOT)) for path in (ROOT/'selected').iterdir() if path.is_dir()}
expected_packages = set(selected.package_path)
assert actual_packages == expected_packages
assert set(validation.candidate_id) == set(selected.candidate_id)
assert (validation.boltzgen_check_status == 'PASS').all()
assert (validation.target_residue_count == 30).all()
assert (validation.target_role == 'geometry_only').all()
assert (~validation.terminal_amide_atomically_verified).all()
print(f'{len(selected)} 个 scaffold 包文件完整且 BoltzGen check 全通过：PASS')

12 个 scaffold 包文件完整且 BoltzGen check 全通过：PASS


## 结论边界

- 该库证明的是数据来源、编号、坐标、规范二硫键和框架多样性可追溯。
- BoltzGen check PASS 只证明 30 残基 target 几何与 scaffold 输入合同可解析；C 端酰胺尚未原子级验证。
- 它没有提供 GLP‑1 结合标签，也没有提供 7–36NH₂ 对 9–36NH₂ 的选择性标签。
- 下一步必须让每个骨架获得相同生成预算，并用统一正/负靶重预测与实验闭环比较。